In [26]:
import refinitiv.data as rd
import pandas as pd
import numpy as np

rd.open_session()


<refinitiv.data.session.Definition object at 0x118b65eb0 {name='workspace'}>

In [27]:
STOCK = "NESTE.HE"
PARAMS = {"SDate": "2024-01-01", "EDate": "2025-12-31", "Frq": "D", "Curn": "EUR"}


In [28]:
daily_field_map = {
    "Price": "TR.PriceClose",
    "MktCap": "TR.CompanyMarketCap",
    "Shares": "TR.SharesOutstanding",
    "REF_PCF": "TR.PriceToCFPerShare",
    "REF_PCF_fiscal": "TR.F.PricetoCFPPerShr",
    "CF": "TR.F.CF",
    "OpCF": "TR.F.NetCashFlowOp",
}

def first_col_is_instrument(frame):
    sample = frame.iloc[:, 0].dropna().astype(str).head(5)
    if sample.empty:
        return False
    return sample.eq(STOCK).all()

def fetch_daily_series(field_code, label):
    raw = rd.get_data(
        universe=[STOCK],
        fields=["TR.PriceClose.date", field_code],
        parameters=PARAMS,
    )

    if "Instrument" in raw.columns or first_col_is_instrument(raw):
        raw = raw.iloc[:, 1:].copy()

    if raw.shape[1] < 2:
        print(f"Kenttä {field_code} ei palauttanut odotettua datasaraketta")
        return pd.DataFrame(columns=["Date", label])

    raw = raw.iloc[:, :2].copy()
    raw.columns = ["Date", label]
    raw["Date"] = pd.to_datetime(raw["Date"], errors="coerce")
    raw[label] = pd.to_numeric(raw[label], errors="coerce")
    raw = raw.dropna(subset=["Date"]).drop_duplicates(subset=["Date"], keep="last")
    return raw.sort_values("Date").reset_index(drop=True)

frames = []
for label, field_code in daily_field_map.items():
    try:
        part = fetch_daily_series(field_code, label)
        frames.append(part)
        print(f"Haettu paivakentta: {label} <- {field_code}")
    except Exception as exc:
        print(f"Ei saatu kenttaa {field_code}: {exc}")

if not frames:
    raise RuntimeError("Yksikaan P/CF-paivakentta ei palautunut")

df = frames[0]
for part in frames[1:]:
    df = df.merge(part, on="Date", how="outer")

for label in daily_field_map:
    if label not in df.columns:
        df[label] = np.nan

df = df.sort_values("Date").reset_index(drop=True)
print(f"{len(df)} riviä")
df.head()


Haettu paivakentta: Price <- TR.PriceClose
Haettu paivakentta: MktCap <- TR.CompanyMarketCap
Haettu paivakentta: Shares <- TR.SharesOutstanding
Haettu paivakentta: REF_PCF <- TR.PriceToCFPerShare
Kenttä TR.F.PricetoCFPPerShr ei palauttanut odotettua datasaraketta
Haettu paivakentta: REF_PCF_fiscal <- TR.F.PricetoCFPPerShr
Haettu paivakentta: CF <- TR.F.CF
Haettu paivakentta: OpCF <- TR.F.NetCashFlowOp
501 riviä


,Date,Price,MktCap,Shares,REF_PCF,REF_PCF_fiscal,CF,OpCF
0,2024-01-02,32.48,24983975163.84,768199747,10.722736,NaN,2527000000,1197000000
1,2024-01-03,31.8,24460911644.400002,768199747,10.498245,NaN,2527000000,1197000000
2,2024-01-04,32.27,24822440841.66,768199747,10.653408,NaN,2527000000,1197000000
3,2024-01-05,32.4,24922438279.199902,768199747,10.696325,NaN,2527000000,1197000000
4,2024-01-08,32.27,24822440841.659901,768199747,10.653408,NaN,2527000000,1197000000


In [29]:
# Itselasketut P/CF ja CF/P
# Forward-fill annual/fiscal kassavirtaluvuille

df["CF_ff"] = df["CF"].ffill()
df["OpCF_ff"] = df["OpCF"].ffill()
df["Shares_ff"] = df["Shares"].ffill()

df["CFPS_from_CF"] = df["CF_ff"] / df["Shares_ff"]
df["CFPS_from_OpCF"] = df["OpCF_ff"] / df["Shares_ff"]

df["PCF_from_CF"] = df["Price"] / df["CFPS_from_CF"]
df["CFP_from_CF"] = df["CFPS_from_CF"] / df["Price"]

df["PCF_from_OpCF"] = df["Price"] / df["CFPS_from_OpCF"]
df["CFP_from_OpCF"] = df["CFPS_from_OpCF"] / df["Price"]

df["PCF_MktCap_div_CF"] = df["MktCap"] / df["CF_ff"]
df["PCF_MktCap_div_OpCF"] = df["MktCap"] / df["OpCF_ff"]

# Refinitivin implied cash flow per share vertailua varten
# Kun REF_PCF on NaN, myös tämä menee NaN:ksi.
df["Implied_CFPS_REF"] = df["Price"] / df["REF_PCF"]
df["Implied_CFPS_fiscal_REF"] = df["Price"] / df["REF_PCF_fiscal"]

df.head()


,Date,Price,MktCap,Shares,REF_PCF,REF_PCF_fiscal,CF,OpCF,CF_ff,OpCF_ff,...,CFPS_from_CF,CFPS_from_OpCF,PCF_from_CF,CFP_from_CF,PCF_from_OpCF,CFP_from_OpCF,PCF_MktCap_div_CF,PCF_MktCap_div_OpCF,Implied_CFPS_REF,Implied_CFPS_fiscal_REF
0,2024-01-02,32.48,24983975163.84,768199747,10.722736,NaN,2527000000,1197000000,2527000000,1197000000,...,3.289509,1.558188,9.873814,0.101278,20.844718,0.047974,9.886812,20.87216,3.029078,NaN
1,2024-01-03,31.8,24460911644.400002,768199747,10.498245,NaN,2527000000,1197000000,2527000000,1197000000,...,3.289509,1.558188,9.667096,0.103444,20.408314,0.049,9.679823,20.435181,3.029078,NaN
2,2024-01-04,32.27,24822440841.66,768199747,10.653408,NaN,2527000000,1197000000,2527000000,1197000000,...,3.289509,1.558188,9.809975,0.101937,20.709946,0.048286,9.822889,20.73721,3.029078,NaN
3,2024-01-05,32.4,24922438279.199902,768199747,10.696325,NaN,2527000000,1197000000,2527000000,1197000000,...,3.289509,1.558188,9.849494,0.101528,20.793377,0.048092,9.862461,20.82075,3.029078,NaN
4,2024-01-08,32.27,24822440841.659901,768199747,10.653408,NaN,2527000000,1197000000,2527000000,1197000000,...,3.289509,1.558188,9.809975,0.101937,20.709946,0.048286,9.822889,20.73721,3.029078,NaN


In [30]:
# Hae kvartaalinen operatiivinen kassavirta ja laske TTM itse jos kentta palautuu

def fetch_quarterly_cf(field_code, value_name):
    try:
        q = rd.get_data(
            universe=[STOCK],
            fields=[f"{field_code}.periodenddate", field_code],
            parameters={"Period": "FQ-19:FQ0", "Frq": "FQ", "Curn": "EUR"}
        )
    except Exception as exc:
        print(f"Ei saatu kenttaa {field_code}: {exc}")
        return pd.DataFrame(columns=["QDate", value_name])

    if "Instrument" in q.columns:
        q = q.drop(columns="Instrument")
    q.columns = ["QDate", value_name]
    q["QDate"] = pd.to_datetime(q["QDate"], errors="coerce")
    q[value_name] = pd.to_numeric(q[value_name], errors="coerce")
    return q.dropna(subset=["QDate"]).sort_values("QDate")

q_opcf_act = fetch_quarterly_cf("TR.F.CashFlowOpActValue", "OpCFAct_Q")
q_opcf = fetch_quarterly_cf("TR.F.NetCashFlowOp", "OpCF_Q")

q = (
    q_opcf_act
    .merge(q_opcf, on="QDate", how="outer")
    .sort_values("QDate")
)

q["TTM_OpCFAct"] = q["OpCFAct_Q"].rolling(4, min_periods=4).sum()
q["TTM_OpCF"] = q["OpCF_Q"].rolling(4, min_periods=4).sum()

print("Kvartaalidata ja itse lasketut TTM-kassavirrat:")
print(q.to_string(index=False))

df = df.sort_values("Date").copy()
q_ttm = (
    q[["QDate", "TTM_OpCFAct", "TTM_OpCF"]]
    .dropna(how="all", subset=["TTM_OpCFAct", "TTM_OpCF"])
    .rename(columns={"QDate": "Date"})
    .sort_values("Date")
)

if not q_ttm.empty:
    df = pd.merge_asof(df, q_ttm, on="Date", direction="backward")
else:
    df["TTM_OpCFAct"] = np.nan
    df["TTM_OpCF"] = np.nan

df["TTM_CFPS_from_OpCFAct"] = df["TTM_OpCFAct"] / df["Shares_ff"]
df["TTM_CFPS_from_OpCF"] = df["TTM_OpCF"] / df["Shares_ff"]

df["Implied_CFPS_self"] = (
    df["TTM_CFPS_from_OpCFAct"]
    .combine_first(df["TTM_CFPS_from_OpCF"])
    .combine_first(df["CFPS_from_OpCF"])
    .combine_first(df["CFPS_from_CF"])
)

df["Implied_CFPS_self_source"] = np.select(
    [
        df["TTM_CFPS_from_OpCFAct"].notna(),
        df["TTM_CFPS_from_OpCFAct"].isna() & df["TTM_CFPS_from_OpCF"].notna(),
        df["TTM_CFPS_from_OpCFAct"].isna() & df["TTM_CFPS_from_OpCF"].isna() & df["CFPS_from_OpCF"].notna(),
        df["TTM_CFPS_from_OpCFAct"].isna() & df["TTM_CFPS_from_OpCF"].isna() & df["CFPS_from_OpCF"].isna() & df["CFPS_from_CF"].notna(),
    ],
    ["TTM_CFPS_from_OpCFAct", "TTM_CFPS_from_OpCF", "CFPS_from_OpCF", "CFPS_from_CF"],
    default=None,
)

df["PCF_self"] = df["Price"] / df["Implied_CFPS_self"]
df["CFP_self"] = df["Implied_CFPS_self"] / df["Price"]

check = df[[
    "Date", "REF_PCF", "REF_PCF_fiscal", "Implied_CFPS_REF", "Implied_CFPS_fiscal_REF",
    "CFPS_from_CF", "CFPS_from_OpCF", "TTM_CFPS_from_OpCFAct", "TTM_CFPS_from_OpCF",
    "Implied_CFPS_self", "Implied_CFPS_self_source", "PCF_self", "CFP_self"
]].copy()
print("\nREF_PCF vs itse laskettu CFPS / P-CF (viimeiset 20):")
print(check.tail(20).to_string(index=False))


Ei saatu kenttaa TR.F.CashFlowOpActValue: Error code -1 | Unable to resolve all requested fields in ['TR.F.CASHFLOWOPACTVALUE.PERIODENDDATE', 'TR.F.CASHFLOWOPACTVALUE']. The formula must contain at least one field or function.
Kvartaalidata ja itse lasketut TTM-kassavirrat:
     QDate OpCFAct_Q     OpCF_Q  TTM_OpCFAct     TTM_OpCF
2021-03-31       NaN -153000000          NaN          NaN
2021-06-30       NaN  567000000          NaN          NaN
2021-09-30       NaN  379000000          NaN          NaN
2021-12-31       NaN 1201000000          NaN 1994000000.0
2022-03-31       NaN -639000000          NaN 1508000000.0
2022-06-30       NaN  254000000          NaN 1195000000.0
2022-09-30       NaN  842000000          NaN 1658000000.0
2022-12-31       NaN  740000000          NaN 1197000000.0
2023-03-31       NaN  377000000          NaN 2213000000.0
2023-06-30       NaN  418000000          NaN 2377000000.0
2023-09-30       NaN  796000000          NaN 2331000000.0
2023-12-31       NaN  6890000

In [31]:
# Trial and error: laita CFPS-ehdokkaat vierekkain ja katso silmalla mikä osuu Refinitiviin
extra_field_candidates = {
    "CFPS_fiscal": "TR.F.CashFlowPerShare",
    "CFOpToShareTtm": "TR.CFOpToShareTtm",
    "OperatingCFTtm": "TR.OperatingCFTtm",
}

for label, field in extra_field_candidates.items():
    try:
        extra = fetch_daily_series(field, label)
        df = df.merge(extra, on="Date", how="left")
        print(f"Haettu lisakandidaatti: {label}")
    except Exception as exc:
        print(f"Ei saatu kenttaa {field}: {exc}")

if "OperatingCFTtm" in df.columns:
    df["OperatingCFTtm_per_share"] = df["OperatingCFTtm"] / df["Shares_ff"]

candidate_cols = [
    "CFPS_from_CF",
    "CFPS_from_OpCF",
    "TTM_CFPS_from_OpCFAct",
    "TTM_CFPS_from_OpCF",
]

if "CFPS_fiscal" in df.columns:
    candidate_cols.insert(0, "CFPS_fiscal")
if "CFOpToShareTtm" in df.columns:
    candidate_cols.insert(1, "CFOpToShareTtm")
if "OperatingCFTtm_per_share" in df.columns:
    candidate_cols.insert(2, "OperatingCFTtm_per_share")

candidate_cols = [c for c in candidate_cols if c in df.columns]

pcf_variant_map = {
    "CFPS_fiscal": "PCF_from_CFPS_fiscal",
    "CFOpToShareTtm": "PCF_from_CFOpToShareTtm",
    "OperatingCFTtm_per_share": "PCF_from_OperatingCFTtm_per_share",
    "CFPS_from_CF": "PCF_from_CF",
    "CFPS_from_OpCF": "PCF_from_OpCF",
    "TTM_CFPS_from_OpCFAct": "PCF_from_TTM_OpCFAct",
    "TTM_CFPS_from_OpCF": "PCF_from_TTM_OpCF",
}
cfp_variant_map = {
    "CFPS_fiscal": "CFP_from_CFPS_fiscal",
    "CFOpToShareTtm": "CFP_from_CFOpToShareTtm",
    "OperatingCFTtm_per_share": "CFP_from_OperatingCFTtm_per_share",
    "CFPS_from_CF": "CFP_from_CF",
    "CFPS_from_OpCF": "CFP_from_OpCF",
    "TTM_CFPS_from_OpCFAct": "CFP_from_TTM_OpCFAct",
    "TTM_CFPS_from_OpCF": "CFP_from_TTM_OpCF",
}

for cfps_col in candidate_cols:
    pcf_col = pcf_variant_map[cfps_col]
    cfp_col = cfp_variant_map[cfps_col]
    df[pcf_col] = df["Price"] / df[cfps_col]
    df[cfp_col] = df[cfps_col] / df["Price"]

pcf_variant_cols = [pcf_variant_map[c] for c in candidate_cols if pcf_variant_map[c] in df.columns]

compare_cols = ["Date", "Price", "REF_PCF"]
if "REF_PCF_fiscal" in df.columns:
    compare_cols.append("REF_PCF_fiscal")
compare_cols.extend(["Implied_CFPS_REF", "Implied_CFPS_fiscal_REF"])
compare_cols.extend(candidate_cols)
compare_cols.extend(pcf_variant_cols)
compare_cols.extend(["PCF_MktCap_div_CF", "PCF_MktCap_div_OpCF"])
compare_cols = [c for c in compare_cols if c in df.columns]

trial_compare = df.loc[df["REF_PCF"].notna(), compare_cols].copy()
numeric_cols = [c for c in trial_compare.columns if c != "Date"]
trial_compare[numeric_cols] = trial_compare[numeric_cols].round(3)

print("P/CF-vertailu eri kassavirtamääritelmillä Refinitivin REF_PCF:ää vasten:")
print(trial_compare.tail(40).to_string(index=False))

availability = []
ref_mask = df["Implied_CFPS_REF"].notna()
for col in candidate_cols:
    availability.append({
        "candidate": col,
        "non_null_total": int(df[col].notna().sum()),
        "non_null_with_ref": int((ref_mask & df[col].notna()).sum()),
    })
availability = pd.DataFrame(availability)
print("\nEi-NaN määrät trial-kandidaateille:")
print(availability.to_string(index=False))

# Vaihda tahan se sarake, joka nayttaa osuvan parhaiten Refinitivin implied CFPS:aan
trial_priority = [
    "CFOpToShareTtm",
    "CFPS_fiscal",
    "OperatingCFTtm_per_share",
    "TTM_CFPS_from_OpCFAct",
    "TTM_CFPS_from_OpCF",
    "CFPS_from_OpCF",
    "CFPS_from_CF",
]
TRIAL_SOURCE_COL = None
for col in trial_priority:
    if col in df.columns and (ref_mask & df[col].notna()).any():
        TRIAL_SOURCE_COL = col
        break

if TRIAL_SOURCE_COL is None:
    for col in trial_priority:
        if col in df.columns and df[col].notna().any():
            TRIAL_SOURCE_COL = col
            break

if TRIAL_SOURCE_COL is None:
    raise ValueError("Yhtään käyttökelpoista trial CFPS -sarjaa ei löytynyt")

print(f"\nKaytossa oleva trial-sarja: {TRIAL_SOURCE_COL}")

df["Implied_CFPS_trial"] = df[TRIAL_SOURCE_COL]
df["Implied_CFPS_trial_source"] = TRIAL_SOURCE_COL
df["PCF_trial"] = df["Price"] / df["Implied_CFPS_trial"]
df["CFP_trial"] = df["Implied_CFPS_trial"] / df["Price"]

print("\nTrial-sarjan viimeiset 20 riviä:")
print(df[["Date", "Implied_CFPS_REF", "Implied_CFPS_trial", "Implied_CFPS_trial_source", "PCF_trial", "CFP_trial"]].tail(20).to_string(index=False))


Kenttä TR.F.CashFlowPerShare ei palauttanut odotettua datasaraketta
Haettu lisakandidaatti: CFPS_fiscal
Kenttä TR.CFOpToShareTtm ei palauttanut odotettua datasaraketta
Haettu lisakandidaatti: CFOpToShareTtm
Kenttä TR.OperatingCFTtm ei palauttanut odotettua datasaraketta
Haettu lisakandidaatti: OperatingCFTtm
P/CF-vertailu eri kassavirtamääritelmillä Refinitivin REF_PCF:ää vasten:
      Date   Price  REF_PCF REF_PCF_fiscal  Implied_CFPS_REF Implied_CFPS_fiscal_REF CFPS_fiscal CFOpToShareTtm OperatingCFTtm_per_share  CFPS_from_CF  CFPS_from_OpCF  TTM_CFPS_from_OpCFAct  TTM_CFPS_from_OpCF PCF_from_CFPS_fiscal PCF_from_CFOpToShareTtm PCF_from_OperatingCFTtm_per_share  PCF_from_CF  PCF_from_OpCF  PCF_from_TTM_OpCFAct  PCF_from_TTM_OpCF  PCF_MktCap_div_CF  PCF_MktCap_div_OpCF
2025-10-31  17.955    8.758            NaN              2.05                     NaN         NaN            NaN                      NaN         1.152            1.54                   <NA>                2.07          

In [32]:
# Implied CF -taulu
base_cols = [
    "Date", "Price", "Shares", "MktCap", "REF_PCF",
    "Implied_CFPS_REF", "Implied_CFPS_fiscal_REF",
    "CFPS_from_CF", "CFPS_from_OpCF", "TTM_CFPS_from_OpCFAct", "TTM_CFPS_from_OpCF",
    "Implied_CFPS_self", "Implied_CFPS_self_source",
    "Implied_CFPS_trial", "Implied_CFPS_trial_source",
    "PCF_self", "CFP_self", "PCF_trial", "CFP_trial",
]
optional_cols = [c for c in [
    "REF_PCF_fiscal", "CFPS_fiscal", "CFOpToShareTtm", "OperatingCFTtm", "OperatingCFTtm_per_share",
    "PCF_from_CFPS_fiscal", "PCF_from_CFOpToShareTtm", "PCF_from_OperatingCFTtm_per_share",
    "PCF_from_TTM_OpCFAct", "PCF_from_TTM_OpCF",
    "CFP_from_CFPS_fiscal", "CFP_from_CFOpToShareTtm", "CFP_from_OperatingCFTtm_per_share",
    "CFP_from_TTM_OpCFAct", "CFP_from_TTM_OpCF",
] if c in df.columns]

implied_cf = df[base_cols[:8] + optional_cols + base_cols[8:]].copy()
implied_cf["REF_CFP"] = 1 / implied_cf["REF_PCF"]
if "REF_PCF_fiscal" in implied_cf.columns:
    implied_cf["REF_CFP_fiscal"] = 1 / implied_cf["REF_PCF_fiscal"]
implied_cf["Self_diff_vs_REF"] = implied_cf["Implied_CFPS_self"] - implied_cf["Implied_CFPS_REF"]
implied_cf["Trial_diff_vs_REF"] = implied_cf["Implied_CFPS_trial"] - implied_cf["Implied_CFPS_REF"]
implied_cf["neg_cfps_self_flag"] = (implied_cf["Implied_CFPS_self"] <= 0).astype("Int64")
implied_cf.loc[implied_cf["Implied_CFPS_self"].isna(), "neg_cfps_self_flag"] = pd.NA
implied_cf["neg_cfps_trial_flag"] = (implied_cf["Implied_CFPS_trial"] <= 0).astype("Int64")
implied_cf.loc[implied_cf["Implied_CFPS_trial"].isna(), "neg_cfps_trial_flag"] = pd.NA
implied_cf["ref_pcf_is_nan"] = implied_cf["REF_PCF"].isna().astype(int)
implied_cf["trial_available"] = implied_cf["Implied_CFPS_trial"].notna().astype(int)
implied_cf["ref_pcf_missing_but_trial_available"] = (
    implied_cf["REF_PCF"].isna() & implied_cf["Implied_CFPS_trial"].notna()
).astype(int)

implied_cf.to_csv("implied_cf.csv", index=False)
print(f"implied_cf.csv ({len(implied_cf)} riviä)")
implied_cf


implied_cf.csv (501 riviä)


,Date,Price,Shares,MktCap,REF_PCF,Implied_CFPS_REF,Implied_CFPS_fiscal_REF,CFPS_from_CF,REF_PCF_fiscal,CFPS_fiscal,...,CFP_trial,REF_CFP,REF_CFP_fiscal,Self_diff_vs_REF,Trial_diff_vs_REF,neg_cfps_self_flag,neg_cfps_trial_flag,ref_pcf_is_nan,trial_available,ref_pcf_missing_but_trial_available
0,2024-01-02,32.48,768199747,24983975163.84,10.722736,3.029078,NaN,3.289509,NaN,NaN,...,0.091379,0.09326,NaN,-0.0611,-0.0611,0,0,0,1,0
1,2024-01-03,31.8,768199747,24460911644.400002,10.498245,3.029078,NaN,3.289509,NaN,NaN,...,0.093333,0.095254,NaN,-0.0611,-0.0611,0,0,0,1,0
2,2024-01-04,32.27,768199747,24822440841.66,10.653408,3.029078,NaN,3.289509,NaN,NaN,...,0.091973,0.093867,NaN,-0.0611,-0.0611,0,0,0,1,0
3,2024-01-05,32.4,768199747,24922438279.199902,10.696325,3.029078,NaN,3.289509,NaN,NaN,...,0.091604,0.09349,NaN,-0.0611,-0.0611,0,0,0,1,0
4,2024-01-08,32.27,768199747,24822440841.659901,10.653408,3.029078,NaN,3.289509,NaN,NaN,...,0.091973,0.093867,NaN,-0.0611,-0.0611,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,2025-12-19,18.5,768243610,14230404573.0,9.023719,2.050153,NaN,1.151978,NaN,NaN,...,0.111873,0.110819,NaN,0.019503,0.019503,0,0,0,1,0
497,2025-12-22,18.685,768243610,14372708618.73,9.113956,2.050153,NaN,1.151978,NaN,NaN,...,0.110766,0.109722,NaN,0.019503,0.019503,0,0,0,1,0
498,2025-12-23,18.885,768243610,14526550830.33,9.21151,2.050153,NaN,1.151978,NaN,NaN,...,0.109593,0.10856,NaN,0.019503,0.019503,0,0,0,1,0
499,2025-12-29,19.24,768243610,14799620755.92,9.384668,2.050153,NaN,1.151978,NaN,NaN,...,0.10757,0.106557,NaN,0.019503,0.019503,0,0,0,1,0


In [33]:
# P/CF-taulu
pcf_cols = ["Date", "Price", "REF_PCF"]
if "REF_PCF_fiscal" in df.columns:
    pcf_cols.append("REF_PCF_fiscal")
pcf_cols.extend([
    "Implied_CFPS_REF", "Implied_CFPS_fiscal_REF",
    "CFPS_fiscal", "CFOpToShareTtm", "OperatingCFTtm_per_share",
    "CFPS_from_CF", "CFPS_from_OpCF", "TTM_CFPS_from_OpCFAct", "TTM_CFPS_from_OpCF",
    "PCF_from_CFPS_fiscal", "PCF_from_CFOpToShareTtm", "PCF_from_OperatingCFTtm_per_share",
    "PCF_from_CF", "PCF_from_OpCF", "PCF_from_TTM_OpCFAct", "PCF_from_TTM_OpCF",
    "PCF_MktCap_div_CF", "PCF_MktCap_div_OpCF",
    "PCF_trial", "PCF_self",
    "Implied_CFPS_trial", "Implied_CFPS_trial_source", "Implied_CFPS_self", "Implied_CFPS_self_source",
])
pcf_cols = [c for c in pcf_cols if c in df.columns]
pcf_compare = df[pcf_cols].copy()
pcf_compare.to_csv("pcf_compare.csv", index=False)
pcf = pcf_compare.copy()
pcf.to_csv("pcf.csv", index=False)
print(f"pcf_compare.csv ({len(pcf_compare)} riviä)")
print(f"pcf.csv ({len(pcf)} riviä)")
pcf_compare


pcf_compare.csv (501 riviä)
pcf.csv (501 riviä)


,Date,Price,REF_PCF,REF_PCF_fiscal,Implied_CFPS_REF,Implied_CFPS_fiscal_REF,CFPS_fiscal,CFOpToShareTtm,OperatingCFTtm_per_share,CFPS_from_CF,...,PCF_from_TTM_OpCFAct,PCF_from_TTM_OpCF,PCF_MktCap_div_CF,PCF_MktCap_div_OpCF,PCF_trial,PCF_self,Implied_CFPS_trial,Implied_CFPS_trial_source,Implied_CFPS_self,Implied_CFPS_self_source
0,2024-01-02,32.48,10.722736,NaN,3.029078,NaN,NaN,NaN,NaN,3.289509,...,<NA>,10.943477,9.886812,20.87216,10.943477,10.943477,2.967978,TTM_CFPS_from_OpCF,2.967978,TTM_CFPS_from_OpCF
1,2024-01-03,31.8,10.498245,NaN,3.029078,NaN,NaN,NaN,NaN,3.289509,...,<NA>,10.714365,9.679823,20.435181,10.714365,10.714365,2.967978,TTM_CFPS_from_OpCF,2.967978,TTM_CFPS_from_OpCF
2,2024-01-04,32.27,10.653408,NaN,3.029078,NaN,NaN,NaN,NaN,3.289509,...,<NA>,10.872722,9.822889,20.73721,10.872722,10.872722,2.967978,TTM_CFPS_from_OpCF,2.967978,TTM_CFPS_from_OpCF
3,2024-01-05,32.4,10.696325,NaN,3.029078,NaN,NaN,NaN,NaN,3.289509,...,<NA>,10.916523,9.862461,20.82075,10.916523,10.916523,2.967978,TTM_CFPS_from_OpCF,2.967978,TTM_CFPS_from_OpCF
4,2024-01-08,32.27,10.653408,NaN,3.029078,NaN,NaN,NaN,NaN,3.289509,...,<NA>,10.872722,9.822889,20.73721,10.872722,10.872722,2.967978,TTM_CFPS_from_OpCF,2.967978,TTM_CFPS_from_OpCF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,2025-12-19,18.5,9.023719,NaN,2.050153,NaN,NaN,NaN,NaN,1.151978,...,<NA>,8.938684,16.079553,12.029082,8.938684,8.938684,2.069656,TTM_CFPS_from_OpCF,2.069656,TTM_CFPS_from_OpCF
497,2025-12-22,18.685,9.113956,NaN,2.050153,NaN,NaN,NaN,NaN,1.151978,...,<NA>,9.02807,16.240349,12.149373,9.02807,9.02807,2.069656,TTM_CFPS_from_OpCF,2.069656,TTM_CFPS_from_OpCF
498,2025-12-23,18.885,9.21151,NaN,2.050153,NaN,NaN,NaN,NaN,1.151978,...,<NA>,9.124705,16.414182,12.279417,9.124705,9.124705,2.069656,TTM_CFPS_from_OpCF,2.069656,TTM_CFPS_from_OpCF
499,2025-12-29,19.24,9.384668,NaN,2.050153,NaN,NaN,NaN,NaN,1.151978,...,<NA>,9.296231,16.722735,12.510246,9.296231,9.296231,2.069656,TTM_CFPS_from_OpCF,2.069656,TTM_CFPS_from_OpCF


In [34]:
# CF/P-taulu
cfp = df[["Date"]].copy()
cfp["REF_CFP"] = 1 / df["REF_PCF"]
if "REF_PCF_fiscal" in df.columns:
    cfp["REF_CFP_fiscal"] = 1 / df["REF_PCF_fiscal"]
for col in [
    "CFP_from_CFPS_fiscal", "CFP_from_CFOpToShareTtm", "CFP_from_OperatingCFTtm_per_share",
    "CFP_from_CF", "CFP_from_OpCF", "CFP_from_TTM_OpCFAct", "CFP_from_TTM_OpCF",
    "CFP_trial", "CFP_self",
]:
    if col in df.columns:
        cfp[col] = df[col]
cfp.to_csv("cfp.csv", index=False)
print(f"cfp.csv ({len(cfp)} riviä)")
cfp


cfp.csv (501 riviä)


,Date,REF_CFP,REF_CFP_fiscal,CFP_from_CFPS_fiscal,CFP_from_CFOpToShareTtm,CFP_from_OperatingCFTtm_per_share,CFP_from_CF,CFP_from_OpCF,CFP_from_TTM_OpCFAct,CFP_from_TTM_OpCF,CFP_trial,CFP_self
0,2024-01-02,0.09326,NaN,NaN,NaN,NaN,0.101278,0.047974,<NA>,0.091379,0.091379,0.091379
1,2024-01-03,0.095254,NaN,NaN,NaN,NaN,0.103444,0.049,<NA>,0.093333,0.093333,0.093333
2,2024-01-04,0.093867,NaN,NaN,NaN,NaN,0.101937,0.048286,<NA>,0.091973,0.091973,0.091973
3,2024-01-05,0.09349,NaN,NaN,NaN,NaN,0.101528,0.048092,<NA>,0.091604,0.091604,0.091604
4,2024-01-08,0.093867,NaN,NaN,NaN,NaN,0.101937,0.048286,<NA>,0.091973,0.091973,0.091973
...,...,...,...,...,...,...,...,...,...,...,...,...
496,2025-12-19,0.110819,NaN,NaN,NaN,NaN,0.062269,0.083237,<NA>,0.111873,0.111873,0.111873
497,2025-12-22,0.109722,NaN,NaN,NaN,NaN,0.061653,0.082412,<NA>,0.110766,0.110766,0.110766
498,2025-12-23,0.10856,NaN,NaN,NaN,NaN,0.061,0.08154,<NA>,0.109593,0.109593,0.109593
499,2025-12-29,0.106557,NaN,NaN,NaN,NaN,0.059874,0.080035,<NA>,0.10757,0.10757,0.10757
